In [ ]:
using Plots, DifferentialEquations, LaTeXStrings
include("imode_sigmoid.jl")
using .ImodeSigmoid

In [ ]:
function SpikingModel!(du,u,p,t)

	Iapp, Ithr, Igain, Ilin = p

	sig_pars = (Ithr=Ithr, Igain=Igain, Ilin=Ilin)
	
	If, Is = u

	du[1] = (-If - Is + Imode_sigmoid_val(If, sig_pars) + Iapp) / τ_f
	du[2] = (-Is + If) / τ_s

end

In [ ]:
τ_f = 0.0001
τ_s = 0.02

Ithr_f = 150e-9
Igain_f = 500e-9
Ilin_f = 400e-9

In [ ]:
IV_Fast(I_eq) = I_eq - Imode_sigmoid_val(I_eq, (Ithr=Ithr_f, Igain=Igain_f, Ilin=Ilin_f))
IV_Slow(I_eq) = 2*I_eq - Imode_sigmoid_val(I_eq, (Ithr=Ithr_f, Igain=Igain_f, Ilin=Ilin_f))

plot([IV_Fast, IV_Slow], 0, 1e-6, legend = false, xlabel="Ieq (A)", ylabel="Iapp (A)", layout = (1, 2), size = (800, 400))

In [ ]:
Tfinal = 2.
tspan = (0.0, Tfinal)

Iapp = 5e-7

x0 = [100e-9, 100e-9]

pars = (Iapp, Ithr_f, Igain_f, Ilin_f)

prob = ODEProblem(SpikingModel!, x0, tspan, pars)
sol = solve(prob, Rodas4P(), abstol=1e-15, reltol=1e-12)
plot(sol, xlabel="Time (s)", ylabel="Current (A)", label=[L"I_f" L"I_s"], size = (800, 400))